# 03b — Two-Stage (lgbm) 임시 단일 노트북

**목적**: 표준 Two-Stage(분류+회귀) 한 번 돌려서 BagZIT plateau(val 0.00571)에 들어오는지 확인. **일회용**.

**구성**:
- 전처리 = `optuna_merged.db / 30-931-001` (yeo-johnson 1차 실험 best) `cleaning_args` 그대로
- Stage 1 (분류): LGBMClassifier, HP = 30-931-001 reg_* params (분류기는 별도 HPO 없었으니 차용)
- Stage 2 (회귀): LGBMRegressor, HP = 30-931-001 reg_* params, **target = log1p(y)** (yeo-johnson 대신)
- 최종 = `P(y>0) × expm1(stage2_pred)`

**학습 입력**: die-level X (broadcast unit y/binary). 최종 die pred → unit sum (BagZIT 패턴 동일).

**격리**: `4_output/_temp/two_stage_temp/` 신규. 모듈 무수정.

**비교 대상**:
- BagZIT plateau best: val 0.005701 (stacking_11base), 단일 0.005708 (zit_only)
- 이 노트북 결과가 plateau 영역(0.0057x)에 들어오면 base 후보로 stacking 추가 가능

## 1. 환경 + import

In [ ]:
import os, sys, json

%run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

## 2. 설정 (30-931-001 best PP + best HP, 변환만 log1p)

30-931-001 best trial #233 (val_rmse=0.001143, yeo-johnson 단위). PP/HP 그대로 가져오고 **target_transform만 yeo-johnson → log1p**로 교체.

In [ ]:
EXP_ID  = 'two-stage-temp-001'
N_FOLDS = 5
CLIP_Y_EXTREME = True
TARGET_TRANSFORM = 'log1p'   # ★ 30-931-001은 yeo-johnson, 우리는 log1p

OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_temp')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS (30-931-001 cleaning_args 그대로) ──
PARAMS = {
    'missing_threshold':          0.9,
    'corr_threshold':             0.98,
    'corr_keep_by':               'target_corr',     # ★ leakage 옵트인 — 30-931-001 best
    'add_indicator':              False,
    'indicator_threshold':        0.15,
    'spatial_max_dist':           1.0,
    'post_impute_corr_threshold': 0.98,
    'post_impute_corr_keep_by':   'std',
}

# ── LGBM HP (30-931-001 reg_* 그대로, Stage 1/2 모두 차용) ──
# study에 분류기 HP 없어서(단일 회귀 study) reg HP를 분류기에도 사용
LGB_HP = dict(
    n_estimators       = 903,
    learning_rate      = 0.005533573630889979,
    num_leaves         = 178,
    max_depth          = 12,
    min_child_samples  = 142,
    subsample          = 0.6105634056703496,
    colsample_bytree   = 0.6738919590642785,
    reg_alpha          = 0.0012579407169105439,
    reg_lambda         = 3.1133747938697414e-07,
    min_split_gain     = 2.1083999800006533e-09,
    path_smooth        = 16.0602712938438,
    random_state       = SEED,
    n_jobs             = -1,
    verbose            = -1,
)

print(f'EXP_ID={EXP_ID} | N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} (30-931-001은 yeo-johnson)')
print(f'OUT_DIR={OUT_DIR}')
print(f'PARAMS keys: {list(PARAMS)}')
print(f'LGB_HP keys: {len(LGB_HP)} (분류/회귀 동일 HP 차용)')

## 3. 데이터 로드 + Y clip

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드] xs={xs.shape}, X feat_cols={len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  y_train: max={y_train_unit.max():.6f}, mean={y_train_unit.mean():.6f}, zero ratio={(y_train_unit==0).mean():.1%}')

## 4. 전처리 (30-931-001 PP) + numpy 변환

In [ ]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  xs_train_die: {xs_train_die.shape}, val: {xs_val_die.shape}, test: {xs_test_die.shape}')

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'

# binary target broadcast (Stage 1 분류용)
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)
print(f'\n  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  y_train_die (broadcasted unit y): mean={y_train_die_broadcast.mean():.6f}')
print(f'  y_bin_die (broadcasted y>0):      pos ratio={y_bin_die_broadcast.mean():.4f}')

## 5. KFold split (BagZIT 노트북과 동일 — shuffle=True, random_state=SEED)

stacking pool 호환 위해 동일 fold split.

In [ ]:
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))
print(f'fold split: {N_FOLDS} folds')
for fi, (tr_idx, vl_idx) in enumerate(FOLDS):
    print(f'  fold {fi+1}: train={len(tr_idx):,} val={len(vl_idx):,}')

## 6. 5-fold Two-Stage 학습 + die-level 캡쳐

각 fold:
1. **Stage 1 분류**: LGBMClassifier, target = (y_die > 0) (broadcast된 binary), 모든 train die
2. **Stage 2 회귀**: LGBMRegressor, target = log1p(y_die), **y_die > 0 die만** (positive die만으로 학습)
3. **최종 die pred** = `prob_die × expm1(reg_die)`
4. **unit pred** = same-unit die-level final 합산

In [ ]:
import time

def _sum_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_unit = np.zeros(n_units)
    np.add.at(pred_unit, inverse, pred_die)
    return pred_unit, unique_units

# die-level 캡쳐
oof_die_prob   = np.full(n_train_die, np.nan)
oof_die_reg    = np.full(n_train_die, np.nan)
oof_die_pred   = np.full(n_train_die, np.nan)

val_die_prob   = np.zeros(n_val_die)
val_die_reg    = np.zeros(n_val_die)
val_die_pred   = np.zeros(n_val_die)

test_die_prob  = np.zeros(n_test_die)
test_die_reg   = np.zeros(n_test_die)
test_die_pred  = np.zeros(n_test_die)

print('=== 5-fold Two-Stage 학습 ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)

    X_tr  = X_train_die[tr_die_mask]
    X_vl  = X_train_die[vl_die_mask]
    y_tr  = y_train_die_broadcast[tr_die_mask]
    yb_tr = y_bin_die_broadcast[tr_die_mask]

    # ── Stage 1 분류 (LGBMClassifier) ──
    clf = lgb.LGBMClassifier(**LGB_HP, objective='binary')
    clf.fit(X_tr, yb_tr)

    # ── Stage 2 회귀 (y>0 die만, log1p target) ──
    pos_mask = y_tr > 0
    if pos_mask.sum() < 100:
        raise RuntimeError(f'fold {fold_idx+1}: y>0 die 수가 너무 적음 ({pos_mask.sum()})')
    y_tr_pos_log = np.log1p(y_tr[pos_mask])
    reg = lgb.LGBMRegressor(**LGB_HP, objective='regression')
    reg.fit(X_tr[pos_mask], y_tr_pos_log)

    # ── 예측 (vl, val, test) ──
    def _predict(Xs):
        prob = clf.predict_proba(Xs)[:, 1]
        prob = np.clip(prob, 0.0, 1.0)
        reg_log = reg.predict(Xs)
        reg_y   = np.clip(np.expm1(reg_log), 0.0, None)   # log1p 역변환 + 음수 clip
        final   = prob * reg_y
        return prob, reg_y, final

    p_vl, r_vl, f_vl = _predict(X_vl)
    p_v,  r_v,  f_v  = _predict(X_val_die)
    p_t,  r_t,  f_t  = _predict(X_test_die)

    # OOF (vl_units)
    oof_die_prob[vl_die_mask] = p_vl
    oof_die_reg[vl_die_mask]  = r_vl
    oof_die_pred[vl_die_mask] = f_vl

    # val/test 5-fold avg
    val_die_prob  += p_v / N_FOLDS
    val_die_reg   += r_v / N_FOLDS
    val_die_pred  += f_v / N_FOLDS
    test_die_prob += p_t / N_FOLDS
    test_die_reg  += r_t / N_FOLDS
    test_die_pred += f_t / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s) — '
          f'fold pos die ratio={pos_mask.mean():.3f}, prob_vl mean={p_vl.mean():.4f}')

assert not np.isnan(oof_die_prob).any(), 'OOF die prob 미커버'
assert not np.isnan(oof_die_reg).any(),  'OOF die reg 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print(f'\n[학습 완료] {time.time()-t0:.0f}s')

## 7. unit aggregate + RMSE

In [ ]:
oof_unit_arr,  oof_unit_ids  = _sum_die_to_unit(oof_die_pred,  uid_train_die)
val_unit_arr,  val_unit_ids  = _sum_die_to_unit(val_die_pred,  uid_val_die)
test_unit_arr, test_unit_ids = _sum_die_to_unit(test_die_pred, uid_test_die)

oof_unit  = pd.Series(oof_unit_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
val_unit  = pd.Series(val_unit_arr,  index=val_unit_ids).reindex(y_val_unit.index)
test_unit = pd.Series(test_unit_arr, index=test_unit_ids).reindex(y_test_unit.index)

def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))

oof_rmse  = _rmse(oof_unit,  y_train_unit)
val_rmse  = _rmse(val_unit,  y_val_unit)
test_rmse = _rmse(test_unit, y_test_unit)

print('=' * 75)
print(f'  Two-Stage temp (lgbm only, 30-931-001 PP/HP, log1p)')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414')
print(f'    Stacking 11-base (val best):       val=0.005701, test=0.008408')
print(f'    reg_only/lgbm:                     val=0.005731, test=0.008429')
print('=' * 75)
if val_rmse < 0.0058:
    print('  → plateau 영역 안. stacking pool에 추가 가능.')
elif val_rmse < 0.006:
    print('  → plateau 근처. residual 패턴이 다르면 stacking에 도움 가능.')
else:
    print('  → plateau 밖. 본격 HPO 또는 구조 수정 필요.')

## 8. 산출물 저장 (`_temp/two_stage_temp/`)

In [ ]:
def _build_die_df(uid_arr, die_id_arr, position_arr, prob, reg, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL:     uid_arr,
        DIE_KEY_COL: die_id_arr,
        'position':  position_arr,
        'prob':      prob,
        'reg':       reg,
        'pred':      pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_prob, oof_die_reg, oof_die_pred, y_train_unit,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_prob, val_die_reg, val_die_pred, y_val_unit,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_prob, test_die_reg, test_die_pred, y_test_unit,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL: unit_pred.index.values,
        'pred':  unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })

_build_unit_df(oof_unit,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit,  y_val_unit ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit, y_test_unit).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

meta = {
    'exp_id':            EXP_ID,
    'model':             'Two-Stage (lgbm clf + lgbm reg) + log1p',
    'target_transform':  TARGET_TRANSFORM,
    'n_folds':           N_FOLDS,
    'oof_rmse':          oof_rmse,
    'val_rmse':          val_rmse,
    'test_rmse':         test_rmse,
    'preprocess_PARAMS': PARAMS,
    'effective_pp_params': pp['effective_params'],
    'lgb_hp':            {k: v for k, v in LGB_HP.items() if k not in ['random_state', 'n_jobs', 'verbose']},
    'pp_source':         'optuna_merged.db / 30-931-001 cleaning_args (yeo-johnson 1차 best)',
    'hp_source':         'optuna_merged.db / 30-931-001 reg_* params (변환만 yeo-johnson → log1p 교체)',
    'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
    'feat_cols_clean_n': len(feat_cols_clean),
    'SEED':              int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:25s}  {sz:>10,.1f} KB')

## 9. 요약

In [ ]:
print('=' * 75)
print(f' Two-Stage temp (lgbm) — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  PP source         : 30-931-001 cleaning_args')
print(f'  HP source         : 30-931-001 reg_* (clf/reg 동일 차용)')
print(f'  target transform  : {TARGET_TRANSFORM}')
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → val/test 가 plateau(0.0057x/0.0084x) 안에 들어오면 stacking 후보.')
print(f'  → residual corr 가 BagZIT 변형들과 다르면 stacking 효과 기대.')
print('=' * 75)